In [1]:
# pobieramy wszystko co zwykle plus pydantica do base model, który pozwala na evaliuacje czy dany tekst ma sens

from pydantic import BaseModel 
from dotenv import load_dotenv
from pypdf import PdfReader 
from openai import OpenAI


In [2]:

load_dotenv(override=True)

myOpenAI = OpenAI()

In [3]:
# pobieramy plik pdf

pdfFile = PdfReader('pzpPDF.pdf')
pdfText = ''
for page in pdfFile.pages:
    text = page.extract_text()
    # robimy sprawdzenie czy string trafił do danego text
    if (text):
        pdfText += text

# pobieramy plik txt

with open('pzp.txt','r',encoding= 'utf-8') as f:
    txtText = f.read()


    print(pdfFile,pdfText)

<pypdf._reader.PdfReader object at 0x000001FFE0382C00> Strona 1
Podstawy zarz■dzania projektami
dr Tomasz Kopczy■ski
Strona 2
Wyzwania
PM
Zarz■dzanie
sytuacjami
kryzysowymi
Zarz■dzanie
zespo■ami
rozproszonymi i
zró■nicowanymi
Zwinne i
elastyczne
zespo■y
Zarz■dzanie
kompetencjami
i emocjami
Strona 3
 PROJEKT– tymczasowe przedsi■wzi■cie, maj■ce na celu
stworzenie szczególnej warto■ci (produkt, efekt), gdzie
tymczasowo■■ oznacza, ■e przedsi■wzi■cie ma okre■lony
pocz■tek i koniec.
 ZARZ■DZANIE PROJEKTEM – dzia■anie, w trakcie którego
osoba kieruj■ca projektem (kierownik projektu) przeprowadza
wraz z zespo■em planowanie oraz realizuje i monitoruje projekt
stosuj■c odpowiednie metody i dokumenty, aby osi■gn■■
wyznaczony cel w okre■lonym czasie, zakresie, kosztach i
osi■gaj■c odpowiedni■ jako■■ ko■cow■ przedsi■wzi■cia.
Strona 4
4 edycja (2009)
460 stron
Kluczowe procesy,
produkty, narz■dzi
i techniki
6 edycja (2017)
760 stron
Elementy
podej■cia
zwinnego
5 edycja (2013)
590 stron
Nowy obszar

In [4]:
# system prompt  

system_prompt = f"""
Jesteś asystentem eksperckim wspierającym użytkownika w zrozumieniu zarządzania projektami.

Odpowiadasz na pytania na podstawie dostarczonych materiałów, szczególnie dotyczących:
- definicji: projektu i zarządzania projektem
- etapów projektu (inicjowanie, planowanie, realizacja, zamknięcie)
- trójkąta projektowego (zakres, czas, budżet, jakość)
- ryzyka, interesariuszy i ról w projekcie

Odpowiadaj jasno, konkretnie i w sposób uporządkowany.
Jeśli to możliwe, podawaj przykłady praktyczne.

Jeśli nie znasz odpowiedzi — powiedz to wprost.

---

## Materiały:

### Dokument:
{txtText}

### PDF:
{pdfText}

---

Na podstawie tego kontekstu odpowiadaj na pytania użytkownika.
"""

In [5]:


# tworzymy obiekt evaluation, którego zadaniem jest przyjąć odpowiedz od modelu ai evaluator który sprawdza czy 1 odpowiedz ze zwykłego modelu
# sepłnia wymagania, które sami utworzymy // nasz obiekt opiera się o biblioteke pydandict i segreguje dane, działa w oparciun mo metode BaseModel
# zwraca dane uszeregowane o 2 pola boolean czyli czy evaluotor dał nam true false i czy dał argumentacje

class Evaluation(BaseModel):
    is_acceptable: bool
    feedback: str

In [6]:
# tworzymy prompt dla evaluatora

# prompt dla evaluatora

evaluator_system_prompt = f"""
Jesteś ewaluatorem odpowiedzi AI.

Twoim zadaniem jest ocenić, czy odpowiedź jest zgodna z dostarczonymi materiałami dotyczącymi zarządzania projektami.

Weź pod uwagę:
- poprawność merytoryczną
- zgodność z koncepcjami (etapy projektu, trójkąt projektowy, ryzyko, interesariusze, role)
- czy odpowiedź jest jasna i użyteczna

Jeśli odpowiedź zawiera błędy, nieścisłości lub wychodzi poza kontekst — oznacz ją jako nieakceptowalną.

---

## Materiały:

### Dokument:
{txtText}

### PDF:
{pdfText}

---

Zwróć WYŁĄCZNIE JSON w formacie:
 is_acceptable (bool), feedback (string)
"""

In [7]:

# teraz tworzymy funkcje która przygotuje pełnie wiadomosci dla evalutora, czyli reply wiadomosc od zwykłego modelu, message nasze pytanie dla modelu i history
# tablice która zwiera historie rozmów 

def evaluator_user_prompt(reply,message,history):
    user_prompt = f'oto pełny przebieg romozwy : \n{history}\n\n'
    user_prompt += f'pytanie uzytkownika: \n{message}\n\n'
    user_prompt += f'odpowiedź agenta: \n{reply}\n\n'
    user_prompt += f'oceń tę odpowież'
    return user_prompt





In [8]:
# piszemy funkcje, która przyjmuje reply message history, uruchanmi evaluator user promtp, tworzy z niego i z evalutor_system promtp tablice message.
#  uruvchamia model evaluator który sprawdza odpowiedz 1 agenta, i przekazuje ta odpowiedz do naszej klasy Evaluationm która ukłąda dane wg schematu

def evaluator(reply,message,history) -> Evaluation:

# tworzymy liste messages która przekazemy do modelu
    messages = [

        {'role':'system','content':evaluator_system_prompt},
        {'role':'user','content': evaluator_user_prompt(reply,message,history)}

    ]
    # teraz budujemy odpowiedz modelu ale zamianst create które daje odpowiedz w stringu, my dajemy metode parse która tworzy obiekt
    # musimy tak zrobić bo nasza odpowiedz idzie do obiektu Evaliton i tam jest ukladana w dane w g schematu który utworzylismy w tej klase obiekcie Evalution
    response = myOpenAI.beta.chat.completions.parse(

        model = 'gpt-4.1-mini',
        messages=messages,
        # dodajemy tez response format, gdzie przekazuejmy maszą klasę Eavaluation
        response_format=Evaluation
        
    )
    # zwracamy response zeby odpowiedz modelu mogla travic do Evaluation
    return response.choices[0].message.parsed

In [9]:

# piszemy funkcje rerun, która odpala się jeśli evaluator uzna 1 odpowiedź modelu za błędną, wówczas utworzony w funkcji model uruchomi się
# funkcja przyjmuje 4 argumenty, reply z 1 modelu message od usera, history oraz feedback z klasy Evaluation

def rerun(reply,message,history,feedback):

    # wpierw definiujemy nowy system prompt powiększony o feedback i
    upgrade_system_prompt = system_prompt + "\n\n poprzednia wiadomość była odrzucona \n"
    upgrade_system_prompt += f'twoja odpowiedż {reply}'
    upgrade_system_prompt += f'feedback {feedback}'

    # teraz budujemy message
    messages = [{'role':'system','content':upgrade_system_prompt}] + history + [{'role':'user',"content": message}]

    # teraz odpalamy model 

    response = myOpenAI.chat.completions.create(
        model='gpt-4.1-mini',
        messages=messages
    )
    return response.choices[0].message.content

In [ ]:

# piszemy funkcje 1 która posiada w sobie model odpowiedzi na pytanie usera jest to 1 odpalana finkcja którj odppowiedz trafia do evaluation
# funkcja przyjmuje historie rozmowy oraz wiadomosc / pytanie usera
def chat(message,history):

    # piszemy warunek, który zweryfikuje czy user dał jakieś instrukcje dla modelu w message u nas warunkiem dla true będzie jesli w message znajdzie się
    # zwrot "dokumentów" wówczas otrzyma dodatkowe instrukcje
    if 'dokumentów' in message:
         system = system_prompt + "\nOdpowiadaj wyłącznie na podstawie dostarczonych dokumentów. Jeśli nie masz informacji — powiedz, że nie wiesz."
    # jesli słowo klucz nie pojawi się w message mamy else i system będzie zawierał jedynie info z system prompt 
    else :
        system = system_prompt

    # tworzymy teraz liste messages  jako 1 dajemy role system u nasz prompr system w zaleznosci od if / else i tego czy user uzyje w message słowa klucz
    messages = [{'role':'system','content':system}] + history + [{'role':'user','content':message}]

    # tworzymy model

    response = myOpenAI.chat.completions.create(

        model = 'gpt-4.1-mini',
        messages=messages
    )
    reply = response.choices[0].message.content

# teraz mają nasza odpowiedz na message z 1 modelu z tej funkcji w zmiennej reply, uruchanmiamy 2 model evalator, przekazujemy mu reply czyli odpowiedz 1 agenta
# message i history, wywolujemy funkcje z 2 modelem a on odpowiedz przekazuje do evaluation czyli naszej klasy pydanitc i ona ocenia czy jest true czy false
    evaluation = evaluator(reply,message,history)

# funkncja if jeslu funkcja evalautor przepuszczona przez model pydantica 
# uzna nasza odpowiedz w zmiennej is_acceptable czyli w tym polu da true to odpalamy komende if która konczy zadanei i daje nam
# odpowiedź
    if evaluation.is_acceptable:
        print("Passed evaluation - returning reply")
        # jeśli nie jest true to 2 funkcja której odpowiedz zostaje przepuszczona rprzez model kalse pydanitoka Evaluation da info ze nie ma akceptu 2 funkcji
    else:
        # wyrzuci nam info o ponownym uruchomioni pole feedback i uruchomi funnkcje 3 rerun która ponownie z updatem co jest zle wywoła pytanie i odpowiedz modelu
        print("Failed evaluation - retrying")
        print(evaluation.feedback)
        # uruchanimamy 3 fubkcje rerun która ponownie odpowiada na pytanie usera jesli 1 model dał zlą odpowiedz a 2 model uznał ją za zła 
        # wówczas uruchanuamy 3 model kjtóry odpowiada na pytanie usera z updetem od 2 funkcju i modelu pudatnic Evaluation
        reply = rerun(reply,message,history,evaluation.feedback) 

    # zwracamy odpowiedz 
    return reply


In [11]:
# wywołanie pytania i funkcji

history = []

msg = "Czym zajmuje się project manager, uzyj dokumentów"

reply = chat(msg, history)

history.append({'role': 'user', 'content': msg})
history.append({'role': 'assistant', 'content': reply})

print(reply)

Passed evaluation - returning reply
Project Manager (Kierownik Projektu) zgodnie z dokumentami zajmuje się:

1. Planowaniem, organizacją oraz realizacją projektu.
2. Monitorowaniem przebiegu projektu, w tym kontrolą czasu, budżetu, zakresu i jakości.
3. Zarządzaniem bieżącymi decyzjami w projekcie, aby osiągnąć wyznaczony cel.
4. Koordynacją działań zespołu projektowego i bezpośrednią odpowiedzialnością przed Właścicielem Projektu oraz Komitetem Sterującym.
5. Zarządzaniem procesami projektu, w tym ryzykiem, komunikacją oraz zmianami.
6. Przygotowywaniem raportów i dokumentacji projektowej.
7. Formalnym zamykaniem projektu po osiągnięciu jego celu.

Przykładowo, Kierownik Projektu prowadzi kick off meeting, zarządza harmonogramem, przydziela zadania, monitoruje postęp, identyfikuje i reaguje na ryzyka oraz raportuje do Komitetu Sterującego.

W skrócie: Project Manager odpowiada za skuteczne przeprowadzenie projektu od inicjowania po zamknięcie, zgodnie z ustalonym zakresem, czasem, bud

In [12]:
msg2 = 'a teraz odpowiedz sam z siebie coś związanego z Planowaniem'

reply2 = chat(msg2,history)

history.append({'role': 'user', 'content': msg2})
history.append({'role': 'assistant', 'content': reply2})

print(reply2)

Passed evaluation - returning reply
Planowanie projektu to kluczowy etap, który wyznacza fundamenty dla całego przedsięwzięcia. W procesie planowania Project Manager definiuje cele projektu, które muszą być jasno określone i mierzalne. Następnie tworzy szczegółowy harmonogram prac, często wykorzystując narzędzia takie jak harmonogram Gantta, który pomaga zobrazować kolejność zadań, czas ich trwania oraz zasoby przypisane do poszczególnych działań.

Ważnym elementem planowania jest podział pracy na mniejsze, łatwiejsze do zarządzania części – tzw. WBS (Work Breakdown Structure). Dzięki temu zespół projektowy może lepiej zrozumieć zakres i odpowiedzialności, a Project Manager może łatwiej monitorować postęp.

Planowanie uwzględnia również identyfikację i ocenę ryzyk, co pozwala przygotować działania zapobiegawcze i reakcje na potencjalne zagrożenia. Ponadto, konieczne jest określenie komunikacji w projekcie, w tym częstotliwości spotkań, raportowania i zarządzania interesariuszami.

Dobr

In [13]:
msg3 = 'a teraz odpowiedz jakaś głupotę'

reply3 = chat(msg3,history)

history.append({'role': 'user', 'content': msg3})
history.append({'role': 'assistant', 'content': reply3})

print(reply3)

Passed evaluation - returning reply
Planowanie w zarządzaniu projektem to proces, który polega na szczegółowym określeniu co, kiedy i przez kogo ma być wykonane, aby osiągnąć cele projektu. Podczas planowania definiuje się cele projektu, zakres prac oraz harmonogram, który pokazuje kolejność i czas trwania poszczególnych zadań. Narzędziem często wykorzystywanym na tym etapie jest harmonogram Gantta, który wizualizuje zadania i ich zależności czasowe.

Kluczowym elementem planowania jest także podział pracy (WBS – Work Breakdown Structure), co pomaga rozłożyć projekt na mniejsze, bardziej zarządzalne części i przypisać odpowiedzialności członkom zespołu projektowego. W trakcie planowania należy także zidentyfikować potencjalne ryzyka i zaplanować działania zapobiegawcze lub minimalizujące ich skutki.

Dzięki solidnemu planowi projekt zyskuje przejrzystość i strukturę, co ułatwia kontrolę postępów i efektywną komunikację wewnątrz zespołu oraz z interesariuszami. Przykładowo, w projekcie 

In [14]:
msg4 = 'a teraz bez żadengo info z dokuemtu'

reply4 = chat(msg4,history)

history.append({'role': 'user', 'content': msg4})
history.append({'role': 'assistant', 'content': reply4})

print(reply4)

Failed evaluation - retrying
Odpowiedź agenta jest humorystyczna i niezgodna z materiałami dotyczącymi zarządzania projektami. Zawiera treści niepoważne i nie merytoryczne, które nie wspierają poprawnego zrozumienia procesu planowania projektu. Zadanie wymagało zgodności z dokumentem, mimo prośby o "głupotę" bez informacji z dokumentu, odpowiedź powinna zachować pewien poziom rzetelności lub wyraźnie zaznaczyć, że to żart. W tej formie nie jest jasna ani użyteczna dla użytkownika, dlatego trzeba ją ocenić jako nieakceptowalną.
Planowanie projektu to jak układanie puzzli na czas — trzeba dobrze przemyśleć, które kawałki pasują razem, a które mogą się zgubić po drodze. Idealny plan to taki, który wytrzyma napór niespodzianek, jak zepsuta kawa w biurze czy nagły atak kota na klawiaturę. Dlatego najlepiej mieć zawsze w zanadrzu plan B, C i D — na wypadek, gdyby pierwotny harmonogram postanowił wziąć wolne. Co więcej, jeśli planujesz za dużo, możesz skończyć z listą zadań dłuższą niż sam pr